## Example Lenient (Content-Based) Scoring

This notebook demonstrates the **lenient** scoring methodology — the counterpart to
the strict, location-keyed method shown in `example_scoring.ipynb`.

Where the strict scorer pairs reactions by their exact
`(Reference, Location.Type, Location.Num)` key, the **lenient** scorer pairs reactions
by **chemical/field similarity** (Hungarian assignment), ignoring `Location` labels.
This is the appropriate methodology for **automated extractors** (LLM / OCR pipelines)
whose location labels cannot be expected to match the ground truth verbatim.

This example uses:
- **`similarity_method="tanimoto"`** — graded Morgan-Tanimoto reaction-SMILES scoring,
  so near-identical structures (ethyl vs butyl ester, differing stereo, salt forms)
  receive partial credit instead of 0.
- **`strip_stereo=False`** — stereochemistry in the incoming SMILES is **kept**, so
  stereo differences are reflected in the score rather than ignored.

The per-field comparison logic (yield, reagents, solvents, time, temperature) is
identical to the strict scorer; only the reaction pairing differs.

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
from lenient_scoring import score_lenient

# Input: a plain list of OPRD-schema reaction dicts (same file used by the strict example).
prediction_path = "../data/validation_reactions.json"

# Output filepath for plots / score files
output_filepath = "../results/"

# Run lenient scoring with graded Tanimoto SMILES similarity, keeping stereochemistry.
score = score_lenient(
    prediction_path,
    similarity_method="tanimoto",   # graded Morgan-Tanimoto reaction-SMILES scoring
    strip_stereo=False,             # keep stereochemistry in the incoming SMILES
    type_aware_matching=True,       # match within the same Location type (Scheme/Table/Experimental)
)
print("Lenient scoring complete.")

### Headline scores

- **Total similarity (matched pairs)** — mean `total_similarity` over all matched reactions.
- **Coverage** — fraction of in-scope gold reactions that were matched (rewards finding
  more of the paper's reactions).
- **Total × coverage** — the per-match quality down-weighted by coverage.

> **⚠️ Caveat on coverage for this example.** `validation_reactions.json` is the
> *focused* human-validation subset — only **10 Schemes, 10 Tables, and 10 Experimentals**
> were re-extracted to spot-check OPRD-100's robustness. It was never intended to cover
> every gold reaction. Because coverage is measured against **all** gold reactions in the
> papers those samples come from, this example shows **low coverage by default** (and hence
> a low `Total × coverage`). This is expected, not a defect — for the focused validation
> set, read the **per-match similarity** as the quality signal, not coverage. A full-paper
> submission would have far higher coverage.

In [ ]:
print(f"Total similarity (matched pairs) : {score.combined_mean:.3f}")
print(f"Total similarity x coverage      : {score.combined_mean_covered:.3f}")
print(f"Coverage (matched / gold)        : {score.num_matched}/{score.num_gold} "
      f"({100 * score.coverage:.1f}%)")
print(f"Reactions extracted / gold       : {score.num_extracted} / {score.num_gold}")
print(f"Similarity method                : {score.similarity_method}")

### Per-field similarity scores

The seven sub-metrics, macro-averaged over all matched pairs.

In [ ]:
per_field = pd.DataFrame(
    {
        "score": {
            "Reaction SMILES": score.reaction_smiles_mean,
            "Reaction steps": score.reaction_steps_mean,
            "Yield": score.yield_mean,
            "Reagent names": score.reagent_name_mean,
            "Reagent amounts": score.reagent_amount_mean,
            "Solvents": score.solvent_mean,
            "Time": score.time_mean,
            "Temperature": score.temperature_mean,
        }
    }
).round(3)
per_field

### Per-type breakdown

The same metrics recomputed for each primary Location type (Scheme / Table /
Experimental / Figure), each with its own matched / gold / predicted counts and coverage.

In [ ]:
breakdown = pd.DataFrame(score.per_type_breakdown()).T
breakdown.round(3)

### Per-match detail

One row per matched reaction pair, with every sub-metric and the indices of the paired
prediction / gold entries. Useful for drilling into individual matches.

In [ ]:
score.per_match_df.head(10)

### Metrics 2-8: Visualisation

Score distributions across all matched pairs, mirroring the strict example's plots.
The combined distribution is plotted first, followed by per-type distributions
(Scheme / Table / Experimental) so they can be compared directly with the strict figures.

In [ ]:
from plotting import Plot

matches = score.per_match_df

shared_title_fontdict = {'fontsize': 16, 'fontweight': 'bold'}
label_fontdict = {'fontsize': 14}
tick_fontdict = {'fontsize': 13.5}
plot = Plot(n_rows=2, n_cols=4, figsize=(10, 6),
            shared_title=f"Lenient Score Distributions ({len(matches)} matched reactions)",
            shared_title_fontdict=shared_title_fontdict,
            label_fontdict=label_fontdict, tick_fontdict=tick_fontdict)

# count of equal reaction step counts
equal_step_counts = matches['reaction_steps_similarity'].value_counts().to_dict()
equal_step_count_values = [equal_step_counts.get(1.0, 0), equal_step_counts.get(0.0, 0)]
# count of equal min/max temperatures
equal_temp_counts = matches['temperature_similarity'].value_counts().to_dict()
equal_temp_count_values = [equal_temp_counts.get(1.0, 0), equal_temp_counts.get(0.0, 0)]

plot.add_hist(row=0, col=0, data=matches['reagent_name_similarity'], bins=20, title="", x_label="Reagent Name Score", y_label="Frequency", alpha=0.7, colour='blue')
plot.add_hist(row=0, col=1, data=matches['reagent_amount_similarity'], bins=20, title="", x_label="Reagent Amount\nScore", y_label="", alpha=0.7, colour='blue')
plot.add_bar(row=0, col=2, categories=["Yes", "No"], values=equal_step_count_values, title="", x_label="Equal reaction step\ncount?", y_label="", alpha=0.7, colour='blue')
plot.add_hist(row=0, col=3, data=matches['reaction_smiles_similarity'], bins=20, title="", x_label="Reaction SMILES\nScore (Tanimoto)", y_label="", alpha=0.7, colour='blue')
plot.add_hist(row=1, col=0, data=matches['solvent_similarity'], bins=20, title="", x_label="Solvents Score", y_label="Frequency", alpha=0.7, colour='blue')
plot.add_hist(row=1, col=1, data=matches['time_similarity'], bins=20, title="", x_label="Time Score", y_label="", alpha=0.7, colour='blue')
plot.add_bar(row=1, col=2, categories=["Yes", "No"], values=equal_temp_count_values, title="", x_label="Equal (min, max)\ntemperatures?", y_label="", alpha=0.7, colour='blue')
plot.add_hist(row=1, col=3, data=matches['yield_similarity'], bins=20, title="", x_label="Yield Data Score", y_label="", alpha=0.7, colour='blue')

plot.plot(savefig=True, filepath=output_filepath + "LenientSimilarityScoreDistributions.png")


# --- Per-type distributions (Scheme / Table / Experimental), mirroring the strict example ---
def plot_type_distributions(df, category):
    sub = df[df['primary_type'] == category]
    if len(sub) == 0:
        print(f"No matched {category} reactions to plot.")
        return
    p = Plot(n_rows=2, n_cols=4, figsize=(10, 6),
             shared_title=f"Lenient Score Distributions ({category} - {len(sub)} reactions)",
             shared_title_fontdict=shared_title_fontdict,
             label_fontdict=label_fontdict, tick_fontdict=tick_fontdict)
    step_counts = sub['reaction_steps_similarity'].value_counts().to_dict()
    step_vals = [step_counts.get(1.0, 0), step_counts.get(0.0, 0)]
    temp_counts = sub['temperature_similarity'].value_counts().to_dict()
    temp_vals = [temp_counts.get(1.0, 0), temp_counts.get(0.0, 0)]
    p.add_hist(row=0, col=0, data=sub['reagent_name_similarity'], bins=20, title="", x_label="Reagent Name Score", y_label="Frequency", alpha=0.7, colour='blue')
    p.add_hist(row=0, col=1, data=sub['reagent_amount_similarity'], bins=20, title="", x_label="Reagent Amount\nScore", y_label="", alpha=0.7, colour='blue')
    p.add_bar(row=0, col=2, categories=["Yes", "No"], values=step_vals, title="", x_label="Equal reaction step\ncount?", y_label="", alpha=0.7, colour='blue')
    p.add_hist(row=0, col=3, data=sub['reaction_smiles_similarity'], bins=20, title="", x_label="Reaction SMILES\nScore (Tanimoto)", y_label="", alpha=0.7, colour='blue')
    p.add_hist(row=1, col=0, data=sub['solvent_similarity'], bins=20, title="", x_label="Solvents Score", y_label="Frequency", alpha=0.7, colour='blue')
    p.add_hist(row=1, col=1, data=sub['time_similarity'], bins=20, title="", x_label="Time Score", y_label="", alpha=0.7, colour='blue')
    p.add_bar(row=1, col=2, categories=["Yes", "No"], values=temp_vals, title="", x_label="Equal (min, max)\ntemperatures?", y_label="", alpha=0.7, colour='blue')
    p.add_hist(row=1, col=3, data=sub['yield_similarity'], bins=20, title="", x_label="Yield Data Score", y_label="", alpha=0.7, colour='blue')
    p.plot(savefig=True, filepath=output_filepath + f"Lenient{category}SimilarityScoreDistributions.png")

for category in ["Scheme", "Table", "Experimental"]:
    plot_type_distributions(matches, category)

### Save the scores

`score.write()` writes `lenient_scores.json` (all aggregate metrics + per-type breakdown)
and `lenient_matches.csv` (the per-match detail) to the output directory.

In [ ]:
scores_path = score.write(output_filepath)
print(f"Wrote {scores_path}")
score.to_dict()